# Data Cleaning for Vienna Stations

This notebook tried to clean the `vienna_stations_all_raw_rbl.csv` file.

**Note: I thought that all the station names containing a standalone 'U' where U-bahn stations. This was sadly not the case, so the the notebook is pretty usless**


In [ ]:
import pandas as pd
import os
import sys
import asyncio

input_file = "data/vienna_stations_all_raw_rbl.csv"

try:
    df = pd.read_csv(input_file, sep=';', encoding='utf-8', on_bad_lines='skip')
    print("Data loaded successfully!")
    print(df.head())
except Exception as e:
    print(f"Error loading data: {e}")

In [ ]:
df.info()

In [ ]:
missing_data = df[df.isnull().any(axis=1)]
print(f"Rows with missing data: {len(missing_data)}")
missing_data.head()

In [ ]:
df_filtered = df[df['StopText'].str.contains(r'\bU\b', regex=True, na=False)]

print(f"Original number of stations: {len(df)}")
print(f"Filtered number of stations (with standalone 'U'): {len(df_filtered)}")
print(df_filtered[['StopText']].head())

output_file = "data/vienna_stations_filtered_u.csv"
df_filtered.to_csv(output_file, index=False)
print(f"Filtered data saved to {output_file}")

## Discovery of the "Starts with 4" Rule

Since the text-based filter above was unreliable (it might miss stations or include wrong ones), I decided to perform a **comprehensive scan of the Realtime API**.

The idea is to take *every* available Station ID from our raw dataset, query the API, and check if it actually serves a U-Bahn line (U1-U6). This is what the `StationsManager` originally did before optimization.

### 1. Setup API Access

In [ ]:
# Add parent directory to path to allow importing realtime_api from parent folder
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

try:
    from wiener_linien_api import WienerLinienAPI
    print("WienerLinienAPI imported successfully.")
except ImportError as e:
    print(f"Error importing WienerLinienAPI: {e}")
    print("Ensure you are running this notebook in the correct environment/path.")

### 2. Define the Scan Logic

We iterate through all RBL IDs, fetch their realtime monitors, and filter for lines starting with "U".

In [ ]:
async def fetch_all_ubahn_data(rbl_ids):
    api = WienerLinienAPI()
    # Reduce batch size slightly to be safe in notebook environment
    batch_size = 100
    all_ubahn_events = []
    
    total_batches = (len(rbl_ids) + batch_size - 1) // batch_size
    print(f"Starting scan of {len(rbl_ids)} stations in {total_batches} batches...")

    for i in range(0, len(rbl_ids), batch_size):
        batch = rbl_ids[i:i+batch_size]
        current_batch = (i // batch_size) + 1
        
        if current_batch % 5 == 0:
            print(f"Processing batch {current_batch}/{total_batches}...")
            
        try:
            data = await api.fetch_batch(batch)
            if data:
                events = api.transform_to_events(data)
                # The core logic: Filter for lines starting with 'U'
                u_events = [e for e in events if e.get('linie', '').strip().upper().startswith('U')]
                if u_events:
                    all_ubahn_events.extend(u_events)
        except Exception as e:
            print(f"Error in batch {current_batch}: {e}")
            
    print(f"\nScan complete. Found {len(all_ubahn_events)} U-Bahn events.")
    return all_ubahn_events

### 3. Execute the Scan
**Warning:** This sends many requests to the API and might take a minute.

In [ ]:
# Prepare the list of IDs from the DataFrame
rbl_ids = df['StopID'].astype(str).str.replace(r'\.0$', '', regex=True).unique().tolist()

# Run the async function
# Note: In modern Jupyter/IPython, top-level await is supported.
result_events = await fetch_all_ubahn_data(rbl_ids)

### 4. Analyze Results

Now we examine the station IDs (RBL numbers) that were returned by this "starts with U" filter. Is there a pattern?

In [ ]:
if result_events:
    df_results = pd.DataFrame(result_events)
    
    # Extract unique RBLs (Station IDs) that were found to be U-Bahn stations
    found_ubahn_ids = df_results['rbl'].unique()
    print(f"Unique U-Bahn Station IDs found: {len(found_ubahn_ids)}")
    
    # Check the first digit of these IDs
    first_digits = set(str(rbl)[0] for rbl in found_ubahn_ids)
    print(f"First digits found in U-Bahn IDs: {first_digits}")
    
    if first_digits == {'4'}:
        print("\n✨ DISCOVERY CONFIRMED: Every single station serving a U-Bahn line has an ID starting with '4'.")
    else:
        print(f"\nDiscovery inconclusive. Found mixed starting digits: {first_digits}")
        
    # Display some examples
    print(df_results[['rbl', 'station', 'linie']].drop_duplicates().head())
else:
    print("No U-Bahn events found. Please check your internet connection or API status.")

### Conclusion

Based on this scan, we can define a simplified rule for our `StationsManager`: **If an ID starts with '4', it is a U-Bahn station.**

This allows us to skip the expensive full-scan in the future and just pre-filter IDs starting with '4'.